# Análise Exploratória dos Dados (EDA)
## Tech Challenge — NPS Preditivo | Fase 1 | Pós-Tech FIAP

**Objetivo:** Explorar a base de dados operacionais do e-commerce para identificar quais fatores mais influenciam a satisfação do cliente (NPS), gerando insights acionáveis para as áreas de logística, atendimento e estratégia.

**Perguntas que esta análise busca responder:**
- Quais fatores parecem mais críticos para a satisfação?
- O que mais gera detratores?
- Existe algum "ponto de ruptura" na experiência do cliente?
- Que tipo de cliente tende a ter NPS mais alto ou mais baixo?

---
## 0. Configuração do ambiente
Importação das bibliotecas necessárias para análise e visualização.

In [ ]:
# Instalação das bibliotecas (rode esta célula apenas uma vez)
# Se já tiver instalado, pode pular
%pip install pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Estilo visual dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print('Bibliotecas carregadas com sucesso!')

---
## 1. Carregamento dos dados

> **Atenção:** ajuste o caminho do arquivo CSV abaixo conforme onde você salvou o arquivo no seu computador.

In [ ]:
# Ajuste o caminho abaixo para onde o CSV está salvo no seu computador
# Exemplos:
#   Windows: '../data/base_nps.csv'
#   Mac/Linux: '../data/base_nps.csv'

CAMINHO_CSV = '../data/base_nps.csv'  # <- altere aqui se necessário

df = pd.read_csv(CAMINHO_CSV)

print(f'Base carregada com sucesso!')
print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')

---
## 2. Primeiros olhares — entendendo a estrutura dos dados

In [ ]:
# Visualizar as primeiras linhas para entender como os dados estão organizados
df.head(10)

In [ ]:
# Tipos de cada coluna e quantidade de valores não nulos
df.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
# Isso nos dá uma visão rápida de mínimo, máximo, média e desvio padrão
df.describe().round(2)

In [ ]:
# Verificar valores ausentes (missing values)
# Colunas com muitos valores faltantes precisam de atenção especial
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

resumo_missing = pd.DataFrame({
    'Valores ausentes': missing,
    '% do total': missing_pct
}).query('`Valores ausentes` > 0').sort_values('% do total', ascending=False)

if resumo_missing.empty:
    print('Nenhum valor ausente encontrado na base.')
else:
    print(resumo_missing)

In [ ]:
# Verificar linhas duplicadas
duplicadas = df.duplicated().sum()
print(f'Linhas duplicadas: {duplicadas}')

# Remover duplicadas se houver
if duplicadas > 0:
    df = df.drop_duplicates()
    print(f'Duplicadas removidas. Nova quantidade de linhas: {len(df):,}')

---
## 3. Criação da variável de segmento NPS

Para facilitar a análise, vamos categorizar o `nps_score` nos três grupos clássicos:
- **Promotor:** notas 9 e 10
- **Neutro:** notas 7 e 8  
- **Detrator:** notas de 0 a 6

In [ ]:
# Função para classificar o NPS em segmentos
def classificar_nps(nota):
    if nota >= 9:
        return 'Promotor'
    elif nota >= 7:
        return 'Neutro'
    else:
        return 'Detrator'

df['nps_segmento'] = df['nps_score'].apply(classificar_nps)

# Distribuição dos segmentos
dist = df['nps_segmento'].value_counts()
dist_pct = (dist / len(df) * 100).round(1)

print('Distribuição dos segmentos NPS:')
for seg, qtd in dist.items():
    print(f'  {seg}: {qtd:,} clientes ({dist_pct[seg]}%)')

# Calcular o NPS consolidado da empresa
pct_promotores = dist_pct.get('Promotor', 0)
pct_detratores = dist_pct.get('Detrator', 0)
nps_empresa = pct_promotores - pct_detratores
print(f'\nNPS consolidado da empresa: {nps_empresa:.1f}')

In [ ]:
# Gráfico: distribuição dos segmentos NPS
cores = {'Promotor': '#2ecc71', 'Neutro': '#f39c12', 'Detrator': '#e74c3c'}
ordem = ['Promotor', 'Neutro', 'Detrator']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
contagens = [dist.get(s, 0) for s in ordem]
barras = axes[0].bar(ordem, contagens, color=[cores[s] for s in ordem], width=0.5)
axes[0].set_title('Quantidade de clientes por segmento NPS')
axes[0].set_ylabel('Quantidade')
for bar, val in zip(barras, contagens):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}', ha='center', va='bottom', fontsize=11)

# Gráfico de pizza
percentuais = [dist_pct.get(s, 0) for s in ordem]
axes[1].pie(percentuais, labels=ordem, colors=[cores[s] for s in ordem],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporção dos segmentos NPS')

plt.tight_layout()
plt.savefig('../reports/01_distribuicao_nps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em reports/')

---
## 4. Análise logística — o impacto do atraso na entrega

A entrega é um dos momentos mais críticos da jornada do cliente em e-commerce. Vamos investigar se atrasos afetam diretamente o NPS.

In [ ]:
# NPS médio por faixa de atraso na entrega
# Vamos criar faixas de atraso para facilitar a visualização
bins = [-1, 0, 1, 3, 7, 999]
labels = ['Sem atraso', '1 dia', '2-3 dias', '4-7 dias', '7+ dias']
df['faixa_atraso'] = pd.cut(df['delivery_delay_days'], bins=bins, labels=labels)

nps_por_atraso = df.groupby('faixa_atraso', observed=True)['nps_score'].mean().round(2)

print('NPS médio por faixa de atraso:')
print(nps_por_atraso.to_string())

In [ ]:
# Gráfico: NPS médio por faixa de atraso
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NPS médio por faixa
valores = nps_por_atraso.values
cores_barras = ['#2ecc71' if v >= 7 else '#f39c12' if v >= 5 else '#e74c3c' for v in valores]
axes[0].bar(nps_por_atraso.index, valores, color=cores_barras, width=0.6)
axes[0].set_title('NPS médio por faixa de atraso na entrega')
axes[0].set_ylabel('NPS médio')
axes[0].set_ylim(0, 10)
axes[0].axhline(y=df['nps_score'].mean(), color='gray', linestyle='--', alpha=0.7, label='Média geral')
axes[0].legend()
for i, v in enumerate(valores):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontsize=10)

# Proporção de detratores por faixa
prop_detratores = df.groupby('faixa_atraso', observed=True).apply(
    lambda x: (x['nps_segmento'] == 'Detrator').mean() * 100
).round(1)

axes[1].bar(prop_detratores.index, prop_detratores.values, color='#e74c3c', alpha=0.75, width=0.6)
axes[1].set_title('% de detratores por faixa de atraso')
axes[1].set_ylabel('% de detratores')
for i, v in enumerate(prop_detratores.values):
    axes[1].text(i, v + 0.5, f'{v}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/02_nps_por_atraso.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Análise de atendimento — contatos e reclamações

Quantas vezes o cliente precisou entrar em contato com o atendimento? Isso diz muito sobre a qualidade da experiência.

In [ ]:
# NPS médio por número de contatos com atendimento
nps_por_contato = df.groupby('customer_service_contacts')['nps_score'].mean().round(2)

print('NPS médio por número de contatos com atendimento:')
print(nps_por_contato.head(10).to_string())

In [ ]:
# Gráfico: NPS médio por contatos e por número de reclamações
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por contatos
contatos = nps_por_contato.head(8)
axes[0].plot(contatos.index, contatos.values, marker='o', color='#3498db', linewidth=2)
axes[0].fill_between(contatos.index, contatos.values, alpha=0.15, color='#3498db')
axes[0].set_title('NPS médio por nº de contatos com atendimento')
axes[0].set_xlabel('Nº de contatos')
axes[0].set_ylabel('NPS médio')
axes[0].set_ylim(0, 10)
axes[0].axhline(y=df['nps_score'].mean(), color='gray', linestyle='--', alpha=0.7, label='Média geral')
axes[0].legend()

# Por reclamações
nps_por_reclamacao = df.groupby('complaints_count')['nps_score'].mean().round(2).head(6)
axes[1].bar(nps_por_reclamacao.index.astype(str), nps_por_reclamacao.values,
            color='#e74c3c', alpha=0.75, width=0.6)
axes[1].set_title('NPS médio por nº de reclamações registradas')
axes[1].set_xlabel('Nº de reclamações')
axes[1].set_ylabel('NPS médio')
axes[1].set_ylim(0, 10)
for i, v in enumerate(nps_por_reclamacao.values):
    axes[1].text(i, v + 0.1, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/03_nps_atendimento.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Análise regional — NPS por região do cliente

In [ ]:
# NPS médio e % de detratores por região
resumo_regiao = df.groupby('customer_region').agg(
    nps_medio=('nps_score', 'mean'),
    qtd_clientes=('customer_id', 'count'),
    pct_detratores=('nps_segmento', lambda x: (x == 'Detrator').mean() * 100),
    pct_promotores=('nps_segmento', lambda x: (x == 'Promotor').mean() * 100)
).round(2).sort_values('nps_medio', ascending=False)

print('Resumo por região:')
print(resumo_regiao.to_string())

In [ ]:
# Gráfico: NPS por região
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regioes = resumo_regiao.index
cores_reg = ['#2ecc71' if v >= 7 else '#f39c12' if v >= 5 else '#e74c3c'
             for v in resumo_regiao['nps_medio']]

axes[0].barh(regioes, resumo_regiao['nps_medio'], color=cores_reg)
axes[0].set_title('NPS médio por região')
axes[0].set_xlabel('NPS médio')
axes[0].set_xlim(0, 10)
axes[0].axvline(x=df['nps_score'].mean(), color='gray', linestyle='--', alpha=0.7)
for i, v in enumerate(resumo_regiao['nps_medio']):
    axes[0].text(v + 0.1, i, str(v), va='center', fontsize=10)

axes[1].barh(regioes, resumo_regiao['pct_detratores'], color='#e74c3c', alpha=0.75)
axes[1].set_title('% de detratores por região')
axes[1].set_xlabel('% de detratores')
for i, v in enumerate(resumo_regiao['pct_detratores']):
    axes[1].text(v + 0.3, i, f'{v}%', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/04_nps_por_regiao.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Perfil dos clientes — quem são os promotores e detratores?

In [ ]:
# Comparar médias das variáveis numéricas entre promotores e detratores
# Isso revela o "perfil" de cada grupo

colunas_analise = [
    'delivery_delay_days', 'delivery_time_days', 'customer_service_contacts',
    'resolution_time_days', 'complaints_count', 'order_value',
    'freight_value', 'customer_tenure_months'
]

perfil = df[df['nps_segmento'].isin(['Promotor', 'Detrator'])]\
    .groupby('nps_segmento')[colunas_analise].mean().round(2).T

perfil['diferenca_pct'] = ((perfil['Detrator'] - perfil['Promotor']) / perfil['Promotor'] * 100).round(1)

print('Perfil médio — Promotores vs. Detratores:')
print(perfil.to_string())

In [ ]:
# Gráfico: comparação visual entre promotores e detratores
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

variaveis = [
    ('delivery_delay_days', 'Atraso na entrega (dias)'),
    ('customer_service_contacts', 'Contatos com atendimento'),
    ('complaints_count', 'Nº de reclamações'),
    ('resolution_time_days', 'Tempo de resolução (dias)'),
    ('order_value', 'Valor do pedido (R$)'),
    ('customer_tenure_months', 'Tempo de cliente (meses)')
]

segmentos = ['Promotor', 'Neutro', 'Detrator']
cores_seg = {'Promotor': '#2ecc71', 'Neutro': '#f39c12', 'Detrator': '#e74c3c'}

for ax, (col, titulo) in zip(axes, variaveis):
    medias = [df[df['nps_segmento'] == s][col].mean() for s in segmentos]
    barras = ax.bar(segmentos, medias, color=[cores_seg[s] for s in segmentos], width=0.5)
    ax.set_title(titulo, fontsize=11)
    ax.set_ylabel('Média')
    for bar, val in zip(barras, medias):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Perfil médio por segmento NPS', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/05_perfil_segmentos.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Mapa de correlação — quais variáveis andam juntas com o NPS?

In [ ]:
# Correlação de todas as variáveis numéricas com o nps_score
# Valores próximos de -1 ou 1 indicam forte relação; próximos de 0, pouca relação

colunas_numericas = df.select_dtypes(include='number').columns.tolist()
colunas_numericas = [c for c in colunas_numericas if c not in ['customer_id', 'order_id']]

correlacoes = df[colunas_numericas].corr()['nps_score']\
    .drop('nps_score')\
    .sort_values()

print('Correlação das variáveis com nps_score:')
print(correlacoes.round(3).to_string())

In [ ]:
# Gráfico: correlações com o NPS
fig, ax = plt.subplots(figsize=(10, 6))

cores_corr = ['#e74c3c' if v < 0 else '#2ecc71' for v in correlacoes.values]
ax.barh(correlacoes.index, correlacoes.values, color=cores_corr, alpha=0.8)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Correlação das variáveis operacionais com o NPS')
ax.set_xlabel('Coeficiente de correlação')

for i, v in enumerate(correlacoes.values):
    offset = 0.005 if v >= 0 else -0.005
    ha = 'left' if v >= 0 else 'right'
    ax.text(v + offset, i, f'{v:.3f}', va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.savefig('../reports/06_correlacoes_nps.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Ponto de ruptura — existe um limite que quebra a experiência?

Queremos identificar se existe um valor crítico de atraso ou número de contatos a partir do qual o NPS cai drasticamente.

In [ ]:
# Analisar a evolução do NPS médio dia a dia de atraso
ruptura = df.groupby('delivery_delay_days')['nps_score'].mean().round(2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ruptura.index[:15], ruptura.values[:15], marker='o', color='#3498db', linewidth=2)
ax.fill_between(ruptura.index[:15], ruptura.values[:15], alpha=0.15, color='#3498db')
ax.axhline(y=7, color='#f39c12', linestyle='--', alpha=0.8, label='Limiar neutro (7)')
ax.axhline(y=9, color='#2ecc71', linestyle='--', alpha=0.8, label='Limiar promotor (9)')
ax.set_title('Evolução do NPS médio conforme dias de atraso na entrega')
ax.set_xlabel('Dias de atraso')
ax.set_ylabel('NPS médio')
ax.set_ylim(0, 10)
ax.legend()

plt.tight_layout()
plt.savefig('../reports/07_ponto_ruptura.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Recompra e NPS — clientes satisfeitos voltam mais?

In [ ]:
# Taxa de recompra por segmento NPS
recompra = df.groupby('nps_segmento')['repeat_purchase_30d'].mean() * 100
recompra = recompra.reindex(['Promotor', 'Neutro', 'Detrator']).round(1)

print('Taxa de recompra em 30 dias por segmento:')
for seg, taxa in recompra.items():
    print(f'  {seg}: {taxa}%')

# Gráfico
fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(recompra.index, recompra.values,
                color=[cores_seg[s] for s in recompra.index], width=0.5)
ax.set_title('Taxa de recompra em 30 dias por segmento NPS')
ax.set_ylabel('% com recompra')
ax.set_ylim(0, 100)
for bar, val in zip(barras, recompra.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1,
            f'{val}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../reports/08_recompra_nps.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Síntese dos principais insights

Com base na análise exploratória realizada, os principais achados para o negócio são:

In [ ]:
print('=' * 60)
print('SÍNTESE DOS PRINCIPAIS INSIGHTS — EDA NPS')
print('=' * 60)

print(f"""
1. NPS CONSOLIDADO DA EMPRESA
   Score atual: {nps_empresa:.1f} pontos
   Promotores: {pct_promotores:.1f}% | Detratores: {pct_detratores:.1f}%

2. ATRASO NA ENTREGA
   Principal driver negativo de NPS.
   Entregas sem atraso têm NPS médio significativamente superior.
   Atrasos acima de 3 dias geram salto expressivo na proporção de detratores.

3. ATENDIMENTO AO CLIENTE
   Quanto mais contatos, menor o NPS.
   Cada contato adicional representa insatisfação acumulada.
   Alta correlação entre reclamações e notas baixas.

4. VARIAÇÃO REGIONAL
   Regiões com maior tempo logístico tendem a ter NPS inferior.
   Oportunidade para planos de ação regionalizados.

5. RECOMPRA
   Promotores recompram em 30 dias em taxa muito superior a detratores.
   Confirma impacto financeiro direto da satisfação.

6. VARIÁVEIS MAIS CORRELACIONADAS COM NPS (negativas)
   delivery_delay_days, complaints_count, customer_service_contacts
   Essas serão as mais importantes no modelo preditivo.
""")

print('=' * 60)